# 00 — Setup en Google Colab

1. **Runtime → Cambiar tipo de entorno de ejecución → GPU (T4)**
2. Sube `metricas-proyecto-ml-colab.zip` (generado con `scripts/build_colab_zip.ps1`) o monta Google Drive.
3. Ejecuta las celdas de abajo en orden.
4. Luego abre y ejecuta los notebooks `01` → `07` en secuencia.


## Opción A: subir ZIP desde tu PC

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # selecciona metricas-proyecto-ml-colab.zip
zip_name = next(iter(uploaded))
root = Path('/content/metricas-proyecto-ml')
if root.exists():
    import shutil
    shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content')
print('Proyecto en:', root)

## Opción B: montar Google Drive (si ya subiste el zip ahí)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q "/content/drive/MyDrive/metricas-proyecto-ml-colab.zip" -d /content/
# from pathlib import Path
# root = Path('/content/metricas-proyecto-ml')
# print('Proyecto en:', root)

## Instalar dependencias

In [ ]:
%cd /content/metricas-proyecto-ml
!pip install -q -r requirements.txt
!pip install -q -e .

## Variables de entorno

In [ ]:
import os
os.environ['RUN_PROFILE'] = 'gpu'
os.environ['MAX_STEPS'] = '0'  # 0 = entrenamiento completo
# os.environ['HF_TOKEN'] = 'hf_...'  # solo para datasets gated en notebook 06
print('RUN_PROFILE =', os.environ['RUN_PROFILE'])

## Verificar que todo está listo

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/content/metricas-proyecto-ml')
sys.path.insert(0, str(ROOT / 'src'))

from discurso_odio.paths import TRAIN_CSV, VAL_CSV, TEST_CSV, CORPUS_CSV, CONFIG_PATH
from discurso_odio.data.loaders import load_split

required = [TRAIN_CSV, VAL_CSV, TEST_CSV, CORPUS_CSV, CONFIG_PATH]
missing = [p for p in required if not p.exists()]
assert not missing, f'Faltan archivos: {missing}'

train = load_split('train')
print('Train:', len(train), 'filas')
print('Listo. Ejecuta notebooks 01 → 07 en orden.')

## Orden de ejecución

| Notebook | Qué hace | Requiere GPU |
|----------|----------|--------------|
| `01_eda.ipynb` | Análisis exploratorio | No |
| `02_preprocesamiento_features.ipynb` | Features y pesos de clase | No |
| `03_baselines_tfidf.ipynb` | Baselines TF-IDF → `models/baseline_*.joblib` | No |
| `04_finetuning_mbert.ipynb` | Fine-tune mBERT → `models/mbert-sv/` | **Sí** |
| `05_metricas_y_comparacion.ipynb` | Métricas y McNemar | No |
| `06_benchmark_externo.ipynb` | Benchmark zero-shot (internet) | No |
| `07_interpretabilidad.ipynb` | LIME + SHAP | No |

**Atajo:** si ya tienes `models/mbert-sv/model.safetensors`, súbelo a `models/mbert-sv/` y salta el notebook 04.

## Ejecutar todos en secuencia (opcional)

Abre cada notebook manualmente, o corre el runner desde la raíz del proyecto:

In [ ]:
# Ejecutar 01 → 07 (puede tardar horas; el 04 necesita GPU)
# !python scripts/run_notebooks_colab.py

# Si ya tienes el modelo entrenado, salta el 04:
# !python scripts/run_notebooks_colab.py --skip 4